LAB 05 Machine Learning with Scikit-learn and From Scratch

In [2]:
import pandas as pd
import numpy as np
import time

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import(
    mean_absolute_error, mean_squared_error, r2_score, accuracy_score, precision_score, recall_score, f1_score
)

In [4]:
df= pd.read_csv("garments_worker_productivity.csv")

In [5]:
print("shape:",df.shape)

print("\nColumns:")
print(df.columns.tolist())

print(df.head())


shape: (1197, 15)

Columns:
['date', 'quarter', 'department', 'day', 'team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers', 'actual_productivity']
       date   quarter  department       day  team  targeted_productivity  \
0  1/1/2015  Quarter1      sweing  Thursday     8                   0.80   
1  1/1/2015  Quarter1  finishing   Thursday     1                   0.75   
2  1/1/2015  Quarter1      sweing  Thursday    11                   0.80   
3  1/1/2015  Quarter1      sweing  Thursday    12                   0.80   
4  1/1/2015  Quarter1      sweing  Thursday     6                   0.80   

     smv     wip  over_time  incentive  idle_time  idle_men  \
0  26.16  1108.0       7080         98        0.0         0   
1   3.94     NaN        960          0        0.0         0   
2  11.41   968.0       3660         50        0.0         0   
3  11.41   968.0       3660         50        0.0         0   

In [6]:
print("\nMissing values:")
print(df.isnull().sum())


Missing values:
date                       0
quarter                    0
department                 0
day                        0
team                       0
targeted_productivity      0
smv                        0
wip                      506
over_time                  0
incentive                  0
idle_time                  0
idle_men                   0
no_of_style_change         0
no_of_workers              0
actual_productivity        0
dtype: int64


In [7]:
df.columns = df.columns.str.strip()

# Convert date column
df["date"] = pd.to_datetime(df["date"], format="mixed")

# Drop date column
df = df.drop(columns=["date"])

# Handle missing values in WIP
df["wip"] = df["wip"].fillna(df["wip"].median())

print("\nMissing values after preprocessing:")
print(df.isnull().sum())

print("\nShape after preprocessing:", df.shape)


Missing values after preprocessing:
quarter                  0
department               0
day                      0
team                     0
targeted_productivity    0
smv                      0
wip                      0
over_time                0
incentive                0
idle_time                0
idle_men                 0
no_of_style_change       0
no_of_workers            0
actual_productivity      0
dtype: int64

Shape after preprocessing: (1197, 14)


In [8]:
df["meetsTarget"] = ( df["actual_productivity"] >= df["targeted_productivity"]).astype(int)

print("\nClassification target:")
print(df["meetsTarget"].value_counts())


Classification target:
meetsTarget
1    875
0    322
Name: count, dtype: int64


In [9]:
X_reg = df.drop(columns=["actual_productivity", "meetsTarget"])
y_reg = df["actual_productivity"]

#classification
X_cls = df.drop(columns=["actual_productivity", "meetsTarget"])
y_cls = df["meetsTarget"]

print("\nRegression X shape:", X_reg.shape)
print("Regression y shape:", y_reg.shape)

print("\nClassification X shape:", X_cls.shape)
print("Classification y shape:", y_cls.shape)


Regression X shape: (1197, 13)
Regression y shape: (1197,)

Classification X shape: (1197, 13)
Classification y shape: (1197,)


In [10]:
categorical_columns = ["quarter", "department", "day"]
numerical_columns = ["team","targeted_productivity","smv","wip","over_time","incentive","idle_time","idle_men","no_of_workers","no_of_style_change"]

print("\n categorical columns:")
print(categorical_columns)
print("\n numerical columns:")
print(numerical_columns)


 categorical columns:
['quarter', 'department', 'day']

 numerical columns:
['team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_workers', 'no_of_style_change']


In [11]:
#train test split
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split( X_reg, y_reg, test_size= 0.20 ,random_state= 42)

X_cls_train, X_cls_test, y_cls_train, y_cls_test = train_test_split (X_cls, y_cls,test_size= 0.20 ,random_state= 42, stratify=y_cls )

print("\nRegression:")
print("Training:", X_reg_train.shape)
print("Testing :", X_reg_test.shape)

print("\nClassification:")
print("Training:", X_cls_train.shape)
print("Testing :", X_cls_test.shape)


Regression:
Training: (957, 13)
Testing : (240, 13)

Classification:
Training: (957, 13)
Testing : (240, 13)


In [12]:
#preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_columns),
        ("cat", OneHotEncoder(handle_unknown="ignore"),categorical_columns)
    ]
)

PART A : LINEAR MODEL

In [15]:
linear_model = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("model", LinearRegression())

    ]
)

#training time
start_time = time.time()
linear_model.fit(X_reg_train,y_reg_train)

training_time_reg = time.time() - start_time

#perdiction time
start_time = time.time()

y_reg_pred = linear_model.predict(X_reg_test)

prediction_time_reg = time.time() - start_time

#evaluating

mae = mean_absolute_error(y_reg_test, y_reg_pred)
mse = mean_squared_error(y_reg_test, y_reg_pred)
r2 = r2_score(y_reg_test, y_reg_pred)

print("MAE:",mae)
print("MSE:",mse)
print("R2:", r2)

print("Training TIme: ", training_time_reg, "seconds")
print("prediction time: ", prediction_time_reg, "seconds")


MAE: 0.10845261337977502
MSE: 0.022087184519101778
R2: 0.16816825663056234
Training TIme:  0.015373706817626953 seconds
prediction time:  0.007711172103881836 seconds


PART A : LOGISTIC REGRESSION

In [16]:
logistic_model = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ]
)

#training time
start_time = time.time()
logistic_model.fit(X_cls_train,y_cls_train)

training_time_cls = time.time() - start_time

#prediction time
start_time = time.time()

y_cls_pred = logistic_model.predict(X_cls_test)

prediction_time_cls = time.time() - start_time


#evaluating
accuracy = accuracy_score(y_cls_test, y_cls_pred)
precision = precision_score(y_cls_test, y_cls_pred, zero_division=0)
recall = recall_score(y_cls_test,y_cls_pred, zero_division=0)
f1 = f1_score(y_cls_test, y_cls_pred, zero_division=0)

print("accuracy:",accuracy)
print("precision:",precision)
print("recall:", recall)
print("f1 score", f1)

print("Training TIme: ", training_time_cls, "seconds")
print("prediction time: ", prediction_time_cls, "seconds")

accuracy: 0.7083333333333334
precision: 0.7611940298507462
recall: 0.8742857142857143
f1 score 0.8138297872340425
Training TIme:  0.06876158714294434 seconds
prediction time:  0.005749940872192383 seconds


PART- B FROM SCRATCH

In [27]:
#converting categorical columns to dummy variables

X = pd.get_dummies(
    X_reg, columns= categorical_columns, drop_first= True
)

#converting boolean columns to integers

X = X.astype(float)

X = X.values

y_reg_array = y_reg.values
y_cls_array = y_cls.values

print("\nFrom-scratch X shape:", X.shape)
print("Regression target shape:", y_reg_array.shape)
print("Classification target shape:", y_cls_array.shape)


From-scratch X shape: (1197, 21)
Regression target shape: (1197,)
Classification target shape: (1197,)


In [28]:
#scratch - train and test split

train_indices = X_reg_train.index
test_indices = X_reg_test.index

train_positions = X_reg.index.get_indexer(train_indices)
test_positions = X_reg.index.get_indexer(test_indices)

X_train = X[train_positions]
X_test = X[test_positions]

y_reg_train_s = y_reg_array[train_positions]
y_reg_test_s = y_reg_array[test_positions]

#classification

train_indices_cls = X_cls_train.index
test_indices_cls = X_cls_test.index

train_positions_cls = X_cls.index.get_indexer(train_indices_cls)
test_positions_cls = X_cls.index.get_indexer(test_indices_cls)

X_train_cls = X[train_positions_cls]
X_test_cls = X[test_positions_cls]

y_cls_train_s = y_cls_array[train_positions_cls]
y_cls_test_s = y_cls_array[test_positions_cls]

print("Regression training samples:", X_train.shape[0])
print("Regression testing samples :", X_test.shape[0])

print("Classification training samples:", X_train_cls.shape[0])
print("Classification testing samples :", X_test_cls.shape[0])

Regression training samples: 957
Regression testing samples : 240
Classification training samples: 957
Classification testing samples : 240


In [29]:
#scratch standardization

#regrssion
mean = X_train.mean(axis = 0)
std = X_train.std(axis = 0)

std[std==0] = 1

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

#addind bias/intercept column
X_train = np.c_[np.ones(X_train.shape[0]),X_train]
X_test = np.c_[np.ones(X_test.shape[0]), X_test]

print("\nX_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

#classification
mean_cls = X_train_cls.mean(axis = 0)
std_cls = X_train_cls.std(axis = 0)

std[std==0] = 1

X_train_cls = (X_train_cls - mean_cls) / std_cls
X_test_cls = (X_test_cls - mean_cls) / std_cls

X_train_cls = np.c_[np.ones(X_train_cls.shape[0]), X_train_cls]
X_test_cls = np.c_[np.ones(X_test_cls.shape[0]), X_test_cls]

print("regression training shape:", X_train.shape)
print("regression testing shape:", X_test_cls.shape)



X_train shape: (957, 22)
X_test shape : (240, 22)
regression training shape: (957, 22)
regression testing shape: (240, 22)


From scratch Linear Regression

In [30]:
start_time = time.time()

#calculating coefficeints
beta = np.linalg.pinv(X_train.T @ X_train) @ X_train.T @ y_reg_train_s

training_time_scratch_reg = time.time() - start_time

#for making predictions
start_time = time.time()
y_reg_pred_s = X_test @ beta
prediction_time_scratch_reg = time.time() - start_time

#calculating metrics from scratch

errors = y_reg_test_s - y_reg_pred_s
mae_scratch = np.mean(np.abs(errors))
mse_scratch = np.mean(errors ** 2)

ss_total = np.sum(
    (y_reg_test_s - np.mean(y_reg_test_s)) ** 2
)

ss_residual = np.sum(errors ** 2 )
r2_scratch = 1 - (ss_residual / ss_total)

print("MAE :", mae_scratch)
print("MSE :", mse_scratch)
print("R2  :", r2_scratch)

print("Training Time  :", training_time_scratch_reg, "seconds")
print("Prediction Time:", prediction_time_scratch_reg, "seconds")

MAE : 0.10845261337977521
MSE : 0.022087184519101802
R2  : 0.16816825663056134
Training Time  : 0.007122516632080078 seconds
Prediction Time: 0.00017213821411132812 seconds


From scratch Logistic Regression

In [31]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500 ,500)))

def train_logistic_regression(X,y, learning_rate = 0.01 , epochs =2000):

    weights = np.zeros(X.shape[1])
    for i in range(epochs):
        #linear prediction
        z= X @ weights

        #probability
        predictions = sigmoid(z)

        #gradient
        gradient = (X.T @ (predictions -y)) / len(y)

        #updating weights
        weights = weights - learning_rate * gradient

    return weights

start_time = time.time()

weights = train_logistic_regression(
    X_train_cls, y_cls_train_s, learning_rate=0.01, epochs =2000
)

training_time_scratch_cls = time.time() - start_time

#prediction
start_time = time.time()

probabilities = sigmoid(X_test @ weights)

y_cls_pred_s = (probabilities >= 0.5).astype(int)

prediction_time_scratch_cls = time.time() - start_time

#matrix

tp = np.sum((y_cls_test_s == 1) & (y_cls_pred_s == 1))
tn = np.sum((y_cls_test_s == 0) & (y_cls_pred_s == 0))
fp = np.sum((y_cls_test_s == 0) & (y_cls_pred_s == 1))
fn = np.sum((y_cls_test_s == 1) & (y_cls_pred_s == 0))

accuracy_scratch = (tp + tn) / len(y_cls_test_s)

precision_scratch = tp /(tp + fp) if (tp + fp) != 0 else 0

recall_scratch = tp / (tp + fn) if (tp + fn) != 0 else 0

f1_scratch = (
    2 * precision_scratch * recall_scratch / (precision_scratch + recall_scratch)
    if (precision_scratch + recall_scratch) != 0
    else 0
)


print("Accuracy :", accuracy_scratch)
print("Precision:", precision_scratch)
print("Recall   :", recall_scratch)
print("F1 Score :", f1_scratch)

print("Training Time  :", training_time_scratch_cls, "seconds")
print("Prediction Time:", prediction_time_scratch_cls, "seconds")

Accuracy : 0.7083333333333334
Precision: 0.7354260089686099
Recall   : 0.9371428571428572
F1 Score : 0.8241206030150754
Training Time  : 0.053376197814941406 seconds
Prediction Time: 0.00019598007202148438 seconds


improving linear regression

In [33]:
start_time = time.time()

lambda_reg = 0.01

I = np.eye(X_train.shape[1])

I[0,0] = 0

beta_opt = np.linalg.pinv(
    X_train.T @ X_train + lambda_reg * I
) @ X_train.T @ y_reg_train_s

optimized_training_time_reg = time.time() - start_time

start_time = time.time()

y_reg_pred_opt = X_test @ beta_opt

optimized_prediction_time_reg = time.time() - start_time

errors_opt = y_reg_test_s - y_reg_pred_opt

mae_opt = np.mean(np.abs(errors_opt))
mse_opt = np.mean(errors_opt ** 2 )

ss_total_opt = np.sum(
    (y_reg_test_s - np.mean(y_reg_test_s)) ** 2
)

ss_res_opt = np.sum(errors_opt ** 2)

r2_opt = 1 - (
    ss_res_opt / ss_total_opt
)


print("Improved Linear Regression")
print("MAE:", mae_opt)
print("MSE:", mse_opt)
print("R2:", r2_opt)
print("Training Time:", optimized_training_time_reg)
print("Prediction Time:", optimized_prediction_time_reg)

Improved Linear Regression
MAE: 0.10845139285145446
MSE: 0.022086944023774764
R2: 0.16817731399353075
Training Time: 0.005037069320678711
Prediction Time: 0.0001399517059326172


In [34]:
settings = [
    (0.01, 2000), (0.05, 3000), (0.1, 3000)
]

best_f1 = -1
best_weights= None
best_lr = None
best_epochs = None

start_time = time.time()

for lr, epochs in settings:

    weights_temp = train_logistic_regression(
        X_train_cls,
        y_cls_train_s,
        learning_rate=lr,
        epochs=epochs
    )

    probabilities_temp = sigmoid(
        X_test_cls @ weights_temp
    )

    predictions_temp = (
        probabilities_temp >= 0.5
    ).astype(int)


    tp_temp = np.sum(
        (y_cls_test_s == 1) &
        (predictions_temp == 1)
    )

    fp_temp = np.sum(
        (y_cls_test_s == 0) &
        (predictions_temp == 1)
    )

    fn_temp = np.sum(
        (y_cls_test_s == 1) &
        (predictions_temp == 0)
    )


    precision_temp = (
        tp_temp / (tp_temp + fp_temp)
        if (tp_temp + fp_temp) != 0 else 0
    )

    recall_temp = (
        tp_temp / (tp_temp + fn_temp)
        if (tp_temp + fn_temp) != 0 else 0
    )

    f1_temp = (
        2 * precision_temp * recall_temp /
        (precision_temp + recall_temp)
        if (precision_temp + recall_temp) != 0
        else 0
    )


    if f1_temp > best_f1:
        best_f1 = f1_temp
        best_weights = weights_temp
        best_lr = lr
        best_epochs = epochs


optimized_training_time_cls = time.time() - start_time


start_time = time.time()

probabilities_opt = sigmoid(
    X_test_cls @ best_weights
)

y_cls_pred_opt = (
    probabilities_opt >= 0.5
).astype(int)

optimized_prediction_time_cls = time.time() - start_time


tp = np.sum(
    (y_cls_test_s == 1) &
    (y_cls_pred_opt == 1)
)

tn = np.sum(
    (y_cls_test_s == 0) &
    (y_cls_pred_opt == 0)
)

fp = np.sum(
    (y_cls_test_s == 0) &
    (y_cls_pred_opt == 1)
)

fn = np.sum(
    (y_cls_test_s == 1) &
    (y_cls_pred_opt == 0)
)


accuracy_opt = (tp + tn) / len(y_cls_test_s)

precision_opt = (
    tp / (tp + fp)
    if (tp + fp) != 0 else 0
)

recall_opt = (
    tp / (tp + fn)
    if (tp + fn) != 0 else 0
)

f1_opt = (
    2 * precision_opt * recall_opt /
    (precision_opt + recall_opt)
    if (precision_opt + recall_opt) != 0
    else 0
)


print("Improved Logistic Regression")
print("Best Learning Rate:", best_lr)
print("Best Epochs:", best_epochs)
print("Accuracy:", accuracy_opt)
print("Precision:", precision_opt)
print("Recall:", recall_opt)
print("F1 Score:", f1_opt)
print("Training Time:", optimized_training_time_cls)
print("Prediction Time:", optimized_prediction_time_cls)

Improved Logistic Regression
Best Learning Rate: 0.01
Best Epochs: 2000
Accuracy: 0.7333333333333333
Precision: 0.7581395348837209
Recall: 0.9314285714285714
F1 Score: 0.8358974358974358
Training Time: 0.21741318702697754
Prediction Time: 7.772445678710938e-05


COMPARISON

In [38]:
regression_comparison = pd.DataFrame({
    "implementation": [
        "Scikit-learn", "manual original", "manual improved"
    ],   
    "MAE": [
        mae,
        mae_scratch,
        mae_opt
    ],
    "MSE": [
        mse,
        mse_scratch,
        mse_opt
    ],
    "R2": [
        r2,
        r2_scratch,
        r2_opt
    ],
    "Training Time": [
        training_time_reg,
        training_time_scratch_reg,
        optimized_training_time_reg
    ],
    "Prediction Time": [
        prediction_time_reg,
        prediction_time_scratch_reg,
        optimized_prediction_time_reg
    ]

})

classification_comparison = pd.DataFrame({
    "Implementation": [
        "Scikit-learn",
        "Manual Original",
        "Manual Improved"
    ],
    "Accuracy": [
        accuracy,
        accuracy_scratch,
        accuracy_opt
    ],
    "Precision": [
        precision,
        precision_scratch,
        precision_opt
    ],
    "Recall": [
        recall,
        recall_scratch,
        recall_opt
    ],
    "F1 Score": [
        f1,
        f1_scratch,
        f1_opt
    ],
    "Training Time": [
        training_time_cls,
        training_time_scratch_cls,
        optimized_training_time_cls
    ],
    "Prediction Time": [
        prediction_time_cls,
        prediction_time_scratch_cls,
        optimized_prediction_time_cls
    ]
})


print("REGRESSION COMPARISON")
display(regression_comparison)

print("CLASSIFICATION COMPARISON")
display(classification_comparison)

REGRESSION COMPARISON


,implementation,MAE,MSE,R2,Training Time,Prediction Time
0,Scikit-learn,0.108453,0.022087,0.168168,0.015374,0.007711
1,manual original,0.108453,0.022087,0.168168,0.007123,0.000172
2,manual improved,0.108451,0.022087,0.168177,0.005037,0.000140


CLASSIFICATION COMPARISON


,Implementation,Accuracy,Precision,Recall,F1 Score,Training Time,Prediction Time
0,Scikit-learn,0.708333,0.761194,0.874286,0.813830,0.068762,0.005750
1,Manual Original,0.708333,0.735426,0.937143,0.824121,0.053376,0.000196
2,Manual Improved,0.733333,0.758140,0.931429,0.835897,0.217413,0.000078


In [39]:
print("FINAL ANALYSIS")
print()

print("Regression:")
print(
    "The manual improved model slightly improved the R2 score "
    "from", round(r2_scratch, 6),
    "to", round(r2_opt, 6)
)
print(
    "The manual improved model also reduced training and prediction time."
)

print()

print("Classification:")
print(
    "The manual improved model increased accuracy from",
    round(accuracy_scratch, 6),
    "to", round(accuracy_opt, 6)
)
print(
    "The F1 score increased from",
    round(f1_scratch, 6),
    "to", round(f1_opt, 6)
)
print(
    "The increase in training time is due to testing "
    "different learning rates and numbers of epochs."
)

FINAL ANALYSIS

Regression:
The manual improved model slightly improved the R2 score from 0.168168 to 0.168177
The manual improved model also reduced training and prediction time.

Classification:
The manual improved model increased accuracy from 0.708333 to 0.733333
The F1 score increased from 0.824121 to 0.835897
The increase in training time is due to testing different learning rates and numbers of epochs.


the scratch implementations were compared with the scikit learn implementations using the same train - test data. 